## Netflix Content Rating Analysis
Explore the distribution of Netflix content ratings.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/processed/netflix_cleaned.csv")
df["rating"].value_counts()

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

In [2]:
import plotly.express as px

ratings = df["rating"].value_counts().reset_index()
ratings.columns = ["Rating", "Titles"]

fig = px.bar(
    ratings, x="Rating", y="Titles",
    color="Titles",
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Netflix Content Rating Distribution",
    height=500, width=850
)

fig.update_traces(texttemplate="%{y:,}", textposition="outside")
fig.write_html("../outputs/rating_distribution.html")
fig.show()

### Content Ratings by Type
Compare content ratings between Movies and TV Shows.

In [3]:
rating_type = df.groupby(["rating", "type"]).size().reset_index(name="Titles")

fig = px.bar(
    rating_type, x="rating", y="Titles", color="type",
    barmode="group",
    color_discrete_sequence=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Content Ratings: Movies vs TV Shows",
    height=500, width=850
)

fig.write_html("../outputs/ratings_by_type.html")
fig.show()

### Most Popular Netflix Genres
Identify the most common genres across Netflix titles.

In [4]:
genres = df["listed_in"].dropna().str.split(",").explode().str.strip()
genres.value_counts().head(10)

listed_in
International Movies        2752
Dramas                      2426
Comedies                    1674
International TV Shows      1349
Documentaries                869
Action & Adventure           859
TV Dramas                    762
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64

In [5]:
genre_counts = genres.value_counts().head(10).rename_axis("Genre").reset_index(name="Titles")

fig = px.bar(
    genre_counts.sort_values("Titles"),
    x="Titles", y="Genre", orientation="h",
    color="Titles",
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Top 10 Netflix Genres",
    height=500, width=850
)

fig.write_html("../outputs/top_10_genres.html")
fig.show()

In [6]:
fig = px.pie(
    genre_counts.head(8),
    names="Genre",
    values="Titles",
    hole=0.45,
    color_discrete_sequence=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Top Netflix Genres",
    height=500,
    width=700
)

fig.update_traces(textposition="inside", textinfo="percent+label")
fig.write_html("../outputs/genre_donut.html")
fig.show()

In [7]:
genre_type = df[["type", "listed_in"]].dropna().copy()
genre_type["Genre"] = genre_type["listed_in"].str.split(",")
genre_type = genre_type.explode("Genre")
genre_type["Genre"] = genre_type["Genre"].str.strip()

top_genres = genres.value_counts().head(8).index
summary = genre_type[genre_type["Genre"].isin(top_genres)]

fig = px.histogram(
    summary, x="Genre", color="type",
    barmode="group", histfunc="count",
    color_discrete_sequence=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Genres: Movies vs TV Shows",
    height=500, width=850
)

fig.write_html("../outputs/genre_type_comparison.html")
fig.show()

In [8]:
fig = px.pie(
    ratings.head(8), names="Rating", values="Titles",
    hole=0.5,
    color_discrete_sequence=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Netflix Content Rating Share",
    height=500, width=700
)

fig.update_traces(textposition="inside", textinfo="percent+label")
fig.write_html("../outputs/rating_donut.html")
fig.show()

In [9]:
rating_pivot = df.pivot_table(
    index="rating", columns="type",
    aggfunc="size", fill_value=0
)

fig = px.imshow(
    rating_pivot,
    text_auto=True,
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Netflix Ratings by Content Type",
    height=550, width=750,
    labels=dict(x="Content Type", y="Rating", color="Titles")
)

fig.write_html("../outputs/ratings_heatmap.html")
fig.show()

In [10]:
print("Most Common Rating:", ratings.iloc[0]["Rating"])
print("Most Popular Genre:", genre_counts.iloc[0]["Genre"])
print("Total Rating Categories:", df["rating"].nunique())
print("Total Titles Analyzed:", len(df))


Most Common Rating: TV-MA
Most Popular Genre: International Movies
Total Rating Categories: 14
Total Titles Analyzed: 8790
